In [1]:
import json
import logging
import warnings
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

C:\Users\Admin\semantic-search-case\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
DATA_DIR = Path("../data")
CACHE_DIR = DATA_DIR / "cache"
CORPUS_PATH = DATA_DIR / "code_corpus.json"
QUERIES_PATH = DATA_DIR / "eval_questions.json"
TOP_K = 3

CACHE_DIR.mkdir(exist_ok=True)

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(f)

with open(QUERIES_PATH, "r", encoding="utf-8") as f:
    queries = json.load(f)

logger.info("Corpus: %d documents", len(corpus))
logger.info("Queries: %d", len(queries))

INFO | Corpus: 200 documents
INFO | Queries: 25


In [3]:
def prepare_text(doc):
    return f"{doc.get('function_name', '')} {doc.get('description', '')} {doc.get('code', '')}"

corpus_texts = [prepare_text(doc) for doc in corpus]
query_texts = [q["query"] for q in queries]

logger.info("Example doc: %s...", corpus_texts[0][:80])
logger.info("Example query: %s", query_texts[0])

INFO | Example doc: verify_jwt_token Проверяет JWT-токен и возвращает payload или причину невалиднос...
INFO | Example query: как проверить, истёк ли токен?


In [4]:
safe_name = MODEL_NAME.replace("/", "_")
corpus_cache = CACHE_DIR / f"{safe_name}_corpus.npy"
queries_cache = CACHE_DIR / f"{safe_name}_queries.npy"

model = SentenceTransformer(MODEL_NAME)

if corpus_cache.exists():
    logger.info("Loading corpus embeddings from cache...")
    corpus_embeddings = np.load(corpus_cache)
else:
    logger.info("Computing corpus embeddings...")
    corpus_embeddings = model.encode(corpus_texts, show_progress_bar=True)
    np.save(corpus_cache, corpus_embeddings)
    logger.info("Saved to cache")

if queries_cache.exists():
    logger.info("Loading query embeddings from cache...")
    query_embeddings = np.load(queries_cache)
else:
    logger.info("Computing query embeddings...")
    query_embeddings = model.encode(query_texts, show_progress_bar=True)
    np.save(queries_cache, query_embeddings)
    logger.info("Saved to cache")

logger.info("Corpus embeddings shape: %s", corpus_embeddings.shape)
logger.info("Query embeddings shape: %s", query_embeddings.shape)

INFO | No device provided, using cpu
INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json "HTTP/1.1 200 OK"
INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO | Loading Sentence

In [7]:
def search_top_k(query_emb, corpus_emb, top_k=3):
    similarities = cosine_similarity([query_emb], corpus_emb)[0]
    return np.argsort(similarities)[::-1][:top_k].tolist()

def precision_at_k(retrieved, correct_id, k=3):
    return 1.0 / k if correct_id in retrieved[:k] else 0.0

def reciprocal_rank(retrieved, correct_id):
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id == correct_id:
            return 1.0 / rank
    return 0.0

In [8]:
precisions = []
mrr_values = []

logger.info("Running search for %d queries...", len(queries))

for idx, q in enumerate(queries):
    correct_id = q["correct_chunk_id"]
    top_indices = search_top_k(query_embeddings[idx], corpus_embeddings)
    retrieved_ids = [corpus[i]["id"] for i in top_indices]

    p3 = precision_at_k(retrieved_ids, correct_id)
    rr = reciprocal_rank(retrieved_ids, correct_id)

    precisions.append(p3)
    mrr_values.append(rr)

    if idx < 5:
        status = "OK" if p3 > 0 else "MISS"
        logger.info("%s | %s: correct=%s, top-3=%s, RR=%.4f",
                     status, q["question_id"], correct_id, retrieved_ids, rr)

INFO | Running search for 25 queries...
INFO | MISS | q_01: correct=func_001, top-3=['func_171', 'func_071', 'func_161'], RR=0.0000
INFO | OK | q_02: correct=func_002, top-3=['func_102', 'func_002', 'func_105'], RR=0.5000
INFO | OK | q_03: correct=func_014, top-3=['func_014', 'func_114', 'func_105'], RR=1.0000
INFO | OK | q_04: correct=func_010, top-3=['func_010', 'func_110', 'func_111'], RR=1.0000
INFO | OK | q_05: correct=func_109, top-3=['func_009', 'func_109', 'func_105'], RR=0.5000


In [9]:
mean_p3 = float(np.mean(precisions))
mean_mrr = float(np.mean(mrr_values))

print("\n" + "=" * 50)
print(f"  Model: {MODEL_NAME}")
print(f"  Precision@{TOP_K}: {mean_p3:.4f}")
print(f"  MRR:          {mean_mrr:.4f}")
print("=" * 50)

metrics = {
    "model": MODEL_NAME,
    "precision_at_3": round(mean_p3, 4),
    "mrr": round(mean_mrr, 4)
}

metrics_path = CACHE_DIR / f"{safe_name}_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

logger.info("Metrics saved to %s", metrics_path)

INFO | Metrics saved to ..\data\cache\paraphrase-multilingual-MiniLM-L12-v2_metrics.json



  Model: paraphrase-multilingual-MiniLM-L12-v2
  Precision@3: 0.2800
  MRR:          0.6067
